# 9장 멀티모달 이해

에이전트가 텍스트와 이미지를 함께 처리합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 9장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
## Ollama의 이미지 처리 문제로 gemma4:e4b-it-qat 사용. 관련 수정 제안 https://github.com/ollama/ollama/pull/16879
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b-it-qat &

---

## 1. 언어 모델 선택

이 노트북은 집필 당시 Ollama에서 관찰된 이미지 처리 문제를 고려해 `gemma4:e4b-it-qat`를 지정합니다. [Ollama 관련 변경 제안](https://github.com/ollama/ollama/pull/16879)

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b-it-qat", think=True)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 텍스트와 이미지를 함께 처리하기

Gemma 4 E4B는 텍스트와 함께 이미지 등 여러 형식의 입력을 처리하는 멀티모달 모델입니다. 아래 대화 템플릿에는 이미지를 작은 영역(패치)으로 나누어 표현하는 특수 토큰과 사용자 질문이 함께 들어 있습니다.

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.6;">
<span style="color:#b392f0;">&lt;bos&gt;</span><span style="color:#f97583;">&lt;|turn&gt;system</span>
You are a helpful assistant.<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="color:#79b8ff;">&lt;|image&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;|image|&gt;&lt;image|&gt;</span>Which animal is on the cover of 'Hands-On Large Language Models'?<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

Ollama에 OpenAI 호환 `messages`를 전달하면 서버가 이 형식으로 변환합니다. 이미지 주소를 전달하는 예입니다.

```json
[
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
                },
            },
            {"type": "text", "text": "What’s in this image?"},
        ],
    }
]
```

이미지 URL을 직접 지원하지 않는 추론 엔진에는 이미지 데이터를 Base64로 인코딩해 전달할 수 있습니다.

```json
[
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": "data:image/png;base64,..."},
            },
            {"type": "text", "text": "What’s in this image?"},
        ],
    }
]
```

`content`는 이제 목록입니다. 한 항목에 `image_url`을, 다른 항목에 `text`를 넣으면 서버가 이미지와 질문을 함께 처리합니다. 이를 위해 `memory.py`에는 이미지 입력을 추가할 매개변수를 만들고, `agent.py`에서는 사용자가 이미지를 제공했을 때만 해당 데이터를 메시지에 포함합니다. 먼저 메모리부터 수정합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import Memory

class MultimodalMemory(Memory):
    """대화 이력을 저장하는 기본 메모리이다."""

    def add(
        self,
        role: str,
        content: str,
        tool_call: dict = None,
        image_data: str = None,
    ):
        """메모리에 메시지를 추가한다."""
        # 이미지
        if image_data:
            is_url = image_data.startswith(("http://", "https://"))
            url = image_data if is_url else f"data:image/png;base64,{image_data}"
            content = [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": content},
            ]

        # 본문 메시지
        message = {"role": role, "content": content}

        # 도구 호출
        if tool_call:
            message["tool_calls"] = [tool_call]

        # 메모리에 메시지 추가
        self.messages.append(message)

`add`에서 이미지가 전달됐을 때 메시지의 `content` 구조를 바꾸는 부분을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch9 import memory_diff; memory_diff

## 3. `TinyAgent` 수정

기존 클래스의 기능을 상속하고 사용자의 작업을 처음 `Memory`에 넣을 때 이미지도 함께 전달하도록 변경합니다.

In [ ]:
from illustrated_agents.chapters import ch6


class TinyAgent(ch6.TinyAgent):
    def run(self, task: str, image_data: str = None) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.memory.add("user", task, image_data=image_data)
        self.trajectory.initialize(task)

        # 여러 단계를 수행하는 반복문
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

아래 비교에서 이미지 매개변수를 받고 메모리로 전달하는 두 줄을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch9 import tinyagents_diff; tinyagents_diff

## 4. 멀티모달 에이전트 실행

구성 요소를 연결해 이미지를 분석할 `TinyAgent`를 생성합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct

# 멀티모달 에이전트
agent = TinyAgent(
    llm=llm, 
    tools=NativeTools(), 
    memory=MultimodalMemory(),
    planner=NativeReAct()
)

책 표지를 내려받아 OpenAI 호환 엔드포인트가 읽을 수 있는 Base64 데이터로 인코딩합니다.

아래 코드가 실제로 내려받는 이미지는 저자들의 이전 책 *Hands-On Large Language Models*의 표지입니다.

In [ ]:
import base64
import httpx

# 이미지를 내려받아 인코딩
image_url = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png"
image_data = base64.b64encode(httpx.get(image_url).content).decode("utf-8")

에이전트에 표지와 함께 그림 속 동물이 무엇인지 묻습니다. 답이 표지와 일치하는지 확인합니다.

In [ ]:
agent.run("Which animal is on the cover of 'Hands-On Large Language Models'?", image_data=image_data)

실행 기록에서 모델이 답변에 이른 과정을 확인합니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

이미지 입력은 실행 기록에서 따로 추적하지 않지만 에이전트의 메모리에는 들어 있습니다.

In [ ]:
from rich.pretty import pprint

pprint(agent.memory.get_messages(), max_string=150)

---

# 구현한 내용

이미지를 처리할 수 있는 Gemma 4를 연결하고, `TinyAgent`가 이미지와 질문을 하나의 메시지로 전달하도록 구현했습니다.

In [ ]:
from illustrated_agents.chapters.ch9 import what_we_built; what_we_built

# 다음 장

[10장](../chapter10/chapter10.ipynb)에서는 파일과 코드 실행 도구를 추가해 코딩 에이전트를 만듭니다. 에이전트가 작업 환경을 변경하기 전에 승인을 받도록 하는 방법도 다룹니다.